# AI_LUNG: Kaggle Training Pipeline

This notebook trains and resumes all 3 stages of the AI_LUNG imaging pipeline on Kaggle GPU. It handles:
1. Restoring training progress checkpoints and patient data splits.
2. Remapping dataset paths in `patient_split.json` dynamically.
3. Running training for Stage 1 (Denoising), Stage 2 (3D Reconstruction), and Stage 3 (Nodule Detection).

## Kaggle Dataset Prerequisites
Before running this notebook, make sure you have attached the following datasets to this notebook:
1. **`ai-lung-dataset`**: Containing the original `manifest.zip` (71.8 GB) and `LIDC-XML-only.zip` (9.5 MB).
2. **`ai-lung-checkpoints`**: Containing your downloaded local `outputs` folder (with `splits/patient_split.json` and checkpoints).

## Step 1: GPU and Environment Verification

In [ ]:
import torch
import os

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Change Accelerator to GPU T4 x2 or GPU P100 in the notebook settings.")

## Step 2: Clone Repository and Navigate

In [ ]:
%cd /kaggle/working
if os.path.exists("/kaggle/working/AI_LUNG"):
    print("Repo already cloned. Pulling updates...")
    %cd /kaggle/working/AI_LUNG
    !git pull origin main
else:
    print("Cloning repository...")
    !git clone https://github.com/KailasVS666/AI_LUNG.git /kaggle/working/AI_LUNG
    %cd /kaggle/working/AI_LUNG

print("Current working directory:", os.getcwd())

## Step 3: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
print("Dependencies installed successfully.")

## Step 4: Restore Checkpoints & Remap Split Paths

This copies your splits and checkpoint files into the working directory and automatically converts Windows/Colab paths in `patient_split.json` to Kaggle input paths.

In [ ]:
import shutil
import json
import re

checkpoint_src = "/kaggle/input/datasets/kailassharji/ai-lung-checkpoints"
working_outputs = "/kaggle/working/AI_LUNG/outputs"

# Restore the entire outputs directory
if os.path.exists(checkpoint_src):
    print("Restoring checkpoints folder to working outputs...")
    shutil.copytree(checkpoint_src, working_outputs, dirs_exist_ok=True)
    print("Checkpoints restored.")
else:
    print(f"WARNING: Checkpoint source not found at {checkpoint_src}.")

# Remap paths in patient_split.json
split_path = f"{working_outputs}/splits/patient_split.json"
if os.path.exists(split_path):
    print("Remapping splits dataset paths to Kaggle dataset root...")
    with open(split_path, 'r') as f:
        splits = json.load(f)
    
    KAGGLE_PREFIX = "/kaggle/input/datasets/kailassharji/ai-lung-dataset/manifest/manifest-1600709154662"
    
    def remap_entry(entry):
        loc = entry.get('file_location', '')
        loc_clean = str(loc).replace('\\', '/')
        match = re.search(r'manifest-1600709154662(.*)', loc_clean)
        if match:
            entry['file_location'] = KAGGLE_PREFIX + match.group(1)
        return entry
        
    for split_key in ('train', 'val', 'test'):
        splits[split_key] = [remap_entry(e) for e in splits[split_key]]
        
    with open(split_path, 'w') as f:
        json.dump(splits, f, indent=2)
    
    print(f"Splits remapped successfully. Sample path: {splits['train'][0]['file_location']}")
else:
    print("ERROR: patient_split.json not found! Running build_splits.py to generate new splits...")
    !python scripts/build_splits.py \
        --dataset-root /kaggle/input/datasets/kailassharji/ai-lung-dataset/manifest/manifest-1600709154662 \
        --metadata-csv /kaggle/input/datasets/kailassharji/ai-lung-dataset/manifest/manifest-1600709154662/metadata.csv \
        --out outputs/splits/patient_split.json

## Step 4.5: Preprocess DICOMs to .npy

Running this pre-converts our active dataset subset (50 train, 10 val, 10 test) to `.npy` files. This saves all the files to the local NVMe SSD on Kaggle, boosting training data loading speed by **10-15x** and preventing 12-hour session timeouts!

In [ ]:
!python scripts/preprocess_to_npy.py --config configs/baseline_kaggle.yaml

## Step 5: Stage 1 — Train/Resume Denoising Model

In [ ]:
# Copy restored files to output_dir designated in config if different
# In configs/baseline_kaggle.yaml, the output dir is '/kaggle/working/outputs/train_runs/denoiser_25d'
# We copy them to ensure scripts pick them up automatically
os.makedirs("/kaggle/working/outputs/train_runs", exist_ok=True)
if os.path.exists(f"{working_outputs}/train_runs/denoiser_25d"):
    shutil.copytree(f"{working_outputs}/train_runs/denoiser_25d", 
                    "/kaggle/working/outputs/train_runs/denoiser_25d", 
                    dirs_exist_ok=True)

!python scripts/train_denoiser_baseline.py --config configs/baseline_kaggle.yaml

## Step 6: Export Denoised Volumes (Stage 1 -> Stage 2 Bridge)

In [ ]:
!python scripts/export_denoised.py --config configs/baseline_kaggle.yaml

## Step 7: Stage 2 — Train/Resume 3D Reconstruction

In [ ]:
if os.path.exists(f"{working_outputs}/train_runs/recon3d"):
    shutil.copytree(f"{working_outputs}/train_runs/recon3d", 
                    "/kaggle/working/outputs/train_runs/recon3d", 
                    dirs_exist_ok=True)

!python scripts/train_recon3d.py --config configs/recon3d_kaggle.yaml

## Step 8: Stage 3 — Train/Resume Nodule Detection

In [ ]:
if os.path.exists(f"{working_outputs}/train_runs/nodule_detection"):
    shutil.copytree(f"{working_outputs}/train_runs/nodule_detection", 
                    "/kaggle/working/outputs/train_runs/nodule_detection", 
                    dirs_exist_ok=True)

!python scripts/train_nodule_detector.py --config configs/nodule_detection_kaggle.yaml

## Step 9: Save and Compress Final Checkpoints

Run this to create a single zip file containing all your updated checkpoints and metrics. You can then download this zip directly from the Kaggle Notebook output tab.

In [ ]:
import shutil
print("Compressing outputs...")
shutil.make_archive("/kaggle/working/outputs_updated", 'zip', "/kaggle/working/outputs")
print("Done. Download 'outputs_updated.zip' from the output files section!")